# Convert retrieved price data into required date format

## First try out with 2017 and 2024 data to work out the kinks
- index: date in format dd-mm-yyyy h-m-s-ms
- column: price excluding taxes per hour of dynamic electricity tarif
- note that this is the price for consumers, not on the market

In [63]:
import pandas as pd
import numpy as np

In [182]:
#first inspect the data:
price2017 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2017.csv", sep = ";", index_col = 0)
price2017.index = price2017.index + ":00"
print(price2017.shape)
print(price2017.head())

(8759, 1)
                     prijs_excl_belastingen
datum                                      
01-01-2017 00:00:00                  0.0420
01-01-2017 01:00:00                  0.0499
01-01-2017 02:00:00                  0.0520
01-01-2017 03:00:00                  0.0410
01-01-2017 04:00:00                  0.0390


In [128]:
price2024 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2024.csv")
price2024['datetime'] = price2024.index
price2024['datetime'] = price2024['datetime'].str.split(";").str[0]
price2024.set_index('datetime', inplace=True)
price2024.head()


,datum;prijs_excl_belastingen
datetime,
2024-01-01 00:00:00,100
2024-01-01 01:00:00,10
2024-01-01 02:00:00,0
2024-01-01 03:00:00,10
2024-01-01 04:00:00,30


# Actual Price Dataframe developed

In [193]:
price2017 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2017.csv", sep = ";")
price2018 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2018.csv", sep = ";")
price2019 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2019.csv", sep = ";")
price2020 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2020.csv", sep = ";")
price2021 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2021.csv", sep = ";")
price2022 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2022.csv", sep = ";")
price2023 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2023.csv", sep = ";")
price2024 = pd.read_csv("PrijsDataElek/jeroen_punt_nl_dynamische_stroomprijzen_jaar_2024.csv", sep = ";")

In [213]:
prices2017_2024 = pd.concat([price2017, price2018, price2019, price2020, price2021, price2022, price2023, price2024], axis = 0)
prices2017_2024["datum"] = prices2017_2024["datum"] + ":00"
prices2017_2024 = pd.DataFrame(prices2017_2024)
prices2017_2024["prijs_excl_belastingen"] = prices2017_2024["prijs_excl_belastingen"].astype(float)
prices2017_2024['datum'] = pd.to_datetime(prices2017_2024['datum'], format='mixed')
prices2017_2024.set_index('datum', inplace=True)
print(prices2017_2024.shape)
print(prices2017_2024.isnull().sum())

(70120, 1)
prijs_excl_belastingen    0
dtype: int64


In [214]:
prices2017_2024[0:6]

,prijs_excl_belastingen
datum,
2017-01-01 00:00:00,0.0420
2017-01-01 01:00:00,0.0499
2017-01-01 02:00:00,0.0520
2017-01-01 03:00:00,0.0410
2017-01-01 04:00:00,0.0390
2017-01-01 05:00:00,0.0359


In [252]:
print(prices2017_2024["prijs_excl_belastingen"].dtype)

float64


# Take code from other scripts to merge with our final dataset offshore

In [202]:
data = pd.read_csv("final_offshore_data_2017_2025.csv")
data["date"] = data['full_datetime'].str.rsplit('-', n=1).str[0]
data[['date', 'hour']] = data['correct_days'].str.rsplit('-', n=1, expand=True)
data['hour'] = data['hour'].astype(int) - 1
data['datetime'] = pd.to_datetime(data['date']) + pd.to_timedelta(data['hour'], unit='H')
data = data[data["date"] < "2025-01-01"]

data['month'] = data['datetime'].dt.to_period('M').astype(str)

start_date = pd.Timestamp('2017-01-01')
data['week_number'] = ((data['datetime'] - start_date).dt.days // 7) + 1

# Set 'datetime' as index for the dataset
data.set_index('datetime', inplace=True)

data["max_capacity"] = data["capacity"] / data["percentage"]
data["log_volume"] = np.log1p(data["volume"])
data["ratio_VC"] = data["volume"] / data["max_capacity"]

data.head()

/var/folders/19/l03rx7hx7313wlqw4t3dy1wm0000gn/T/ipykernel_81882/4109503100.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  data['datetime'] = pd.to_datetime(data['date']) + pd.to_timedelta(data['hour'], unit='H')


,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor,correct_days,date,month,week_number,max_capacity,log_volume,ratio_VC
datetime,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,20170101,0,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,873501,873501,1.014165,0,0,2017-01-01-01,2017-01-01,2017-01,1,861300.638701,13.680266,1.014165
2017-01-01 01:00:00,20170101,1,210.905296,87.142857,90.714286,10199.75,96.142857,2017-01-01-02,883749,883749,1.026065,0,0,2017-01-01-02,2017-01-01,2017-01,1,861299.242185,13.691929,1.026065
2017-01-01 02:00:00,20170101,2,208.585001,89.285714,87.857143,10191.50,96.000000,2017-01-01-03,872500,872500,1.013004,0,0,2017-01-01-03,2017-01-01,2017-01,1,861299.705697,13.679119,1.013004
2017-01-01 03:00:00,20170101,3,209.977979,90.000000,90.000000,10182.25,96.142857,2017-01-01-04,889750,889750,1.033031,0,0,2017-01-01-04,2017-01-01,2017-01,1,861300.397937,13.698697,1.033031
2017-01-01 04:00:00,20170101,4,208.541568,89.285714,87.142857,10176.25,95.571429,2017-01-01-05,893251,893251,1.037095,0,0,2017-01-01-05,2017-01-01,2017-01,1,861301.078959,13.702624,1.037095


In [ ]:
Elek_price = pd.concat([data, prices2017_2024], axis=1)
print(Elek_price.shape) #length corresponds with length original dataframe

(70128, 21)


In [243]:
Elek_price.head()

,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,...,emission,emissionfactor,correct_days,date,month,week_number,max_capacity,log_volume,ratio_VC,prijs_excl_belastingen
2017-01-01 00:00:00,20170101,0,213.586816,85.714286,86.428571,10206.75,95.714286,2017-01-01-01,873501,873501,...,0,0,2017-01-01-01,2017-01-01,2017-01,1,861300.638701,13.680266,1.014165,0.0420
2017-01-01 01:00:00,20170101,1,210.905296,87.142857,90.714286,10199.75,96.142857,2017-01-01-02,883749,883749,...,0,0,2017-01-01-02,2017-01-01,2017-01,1,861299.242185,13.691929,1.026065,0.0499
2017-01-01 02:00:00,20170101,2,208.585001,89.285714,87.857143,10191.50,96.000000,2017-01-01-03,872500,872500,...,0,0,2017-01-01-03,2017-01-01,2017-01,1,861299.705697,13.679119,1.013004,0.0520
2017-01-01 03:00:00,20170101,3,209.977979,90.000000,90.000000,10182.25,96.142857,2017-01-01-04,889750,889750,...,0,0,2017-01-01-04,2017-01-01,2017-01,1,861300.397937,13.698697,1.033031,0.0410
2017-01-01 04:00:00,20170101,4,208.541568,89.285714,87.142857,10176.25,95.571429,2017-01-01-05,893251,893251,...,0,0,2017-01-01-05,2017-01-01,2017-01,1,861301.078959,13.702624,1.037095,0.0390


In [ ]:
print(Elek_price.index.dtype)
print(prices2017_2024.index.dtype)
print(Elek_price["year_mon_day"].dtype)
print(Elek_price["hour"].dtype)

# Onshore data including price

In [253]:
data_onsh = pd.read_csv("final_onshore_data_2017_2025.csv")

In [255]:
data_onsh["date"] = data_onsh['full_datetime'].str.rsplit('-', n=1).str[0]
data_onsh[['date', 'hour']] = data_onsh['correct_days'].str.rsplit('-', n=1, expand=True)
data_onsh['hour'] = data_onsh['hour'].astype(int) - 1
data_onsh['datetime'] = pd.to_datetime(data_onsh['date']) + pd.to_timedelta(data_onsh['hour'], unit='H')
data_onsh = data_onsh[data_onsh["date"] < "2025-01-01"]

data_onsh['month'] = data_onsh['datetime'].dt.to_period('M').astype(str)

start_date = pd.Timestamp('2017-01-01')
data_onsh['week_number'] = ((data_onsh['datetime'] - start_date).dt.days // 7) + 1

# Set 'datetime' as index for the dataset
data_onsh.set_index('datetime', inplace=True)

data_onsh["max_capacity"] = data_onsh["capacity"] / data_onsh["percentage"]
data_onsh["log_volume"] = np.log1p(data_onsh["volume"])
data_onsh["ratio_VC"] = data_onsh["volume"] / data_onsh["max_capacity"]

data_onsh.head()

/var/folders/19/l03rx7hx7313wlqw4t3dy1wm0000gn/T/ipykernel_81882/1349340298.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  data_onsh['datetime'] = pd.to_datetime(data_onsh['date']) + pd.to_timedelta(data_onsh['hour'], unit='H')


,year_mon_day,hour,wind_dir_avg_10,wind_speed_h_avg,wind_speed_avg_10,air_pressure,humidity,full_datetime,capacity,volume,percentage,emission,emissionfactor,correct_days,date,month,week_number,max_capacity,log_volume,ratio_VC
datetime,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,20170101,0,207.708194,49.666667,49.666667,10234.526316,98.076923,2017-01-01-01,679334,679334,0.788730,0,0,2017-01-01-01,2017-01-01,2017-01,1,861301.051331,13.428870,0.788730
2017-01-01 01:00:00,20170101,1,205.010321,50.000000,51.333333,10227.789474,98.153846,2017-01-01-02,677462,677462,0.786558,0,0,2017-01-01-02,2017-01-01,2017-01,1,861299.514778,13.426110,0.786558
2017-01-01 02:00:00,20170101,2,202.701006,51.666667,51.000000,10219.473684,98.230769,2017-01-01-03,653746,653746,0.759025,0,0,2017-01-01-03,2017-01-01,2017-01,1,861297.084050,13.390476,0.759025
2017-01-01 03:00:00,20170101,3,201.007553,52.333333,54.666667,10211.368421,98.038462,2017-01-01-04,705882,705882,0.819552,0,0,2017-01-01-04,2017-01-01,2017-01,1,861302.267903,13.467205,0.819552
2017-01-01 04:00:00,20170101,4,200.325015,52.666667,53.333333,10203.526316,97.461538,2017-01-01-05,716738,716738,0.832158,0,0,2017-01-01-05,2017-01-01,2017-01,1,861300.347955,13.482467,0.832158


In [256]:
Elek_price_onsh = pd.concat([data_onsh, prices2017_2024], axis=1)
print(Elek_price_onsh.shape) #length corresponds with length original dataframe

(70128, 21)


In [257]:
Elek_price_onsh.to_csv("Elek_price_onsh.csv")

# Inputing Missing Values
- 8 missing values, all in March, all at 2:00 
- each year has one but no consistent date
- missing values are an average of the price of the two surrounding hours i.e. 1:00 and 3:00


In [250]:
print(Elek_price.isnull().sum())

year_mon_day                0
hour                        0
wind_dir_avg_10             0
wind_speed_h_avg            0
wind_speed_avg_10           0
air_pressure                0
humidity                    0
full_datetime               0
capacity                    0
volume                      0
percentage                  0
emission                    0
emissionfactor              0
correct_days                0
date                        0
month                       0
week_number                 0
max_capacity              463
log_volume                  0
ratio_VC                  463
prijs_excl_belastingen      0
dtype: int64


In [245]:
missing_values = Elek_price[Elek_price["prijs_excl_belastingen"].isnull()]
print(missing_values)

                     year_mon_day  hour  wind_dir_avg_10  wind_speed_h_avg  \
2017-03-26 02:00:00      20170326     2        54.284128         60.714286   
2018-03-25 02:00:00      20180325     2        17.742821         41.333333   
2019-03-31 02:00:00      20190331     2        28.038266         64.000000   
2020-03-29 02:00:00      20200329     2        12.642947        124.000000   
2021-03-28 02:00:00      20210328     2       212.152009         90.000000   
2022-03-27 02:00:00      20220327     2        37.246338         46.666667   
2023-03-26 02:00:00      20230326     2       351.349030         37.333333   
2024-03-31 02:00:00      20240331     2        58.997918         38.666667   

                     wind_speed_avg_10  air_pressure   humidity  \
2017-03-26 02:00:00          60.000000      10266.50  91.428571   
2018-03-25 02:00:00          40.000000      10057.25  95.142857   
2019-03-31 02:00:00          68.000000      10236.50  86.714286   
2020-03-29 02:00:00         1

In [246]:
Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20170326) & 
                                                (Elek_price["hour"] == 2), 
                                                0.0277,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20180325) & 
                                                (Elek_price["hour"] == 2), 
                                                0.04195,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20190331) & 
                                                (Elek_price["hour"] == 2), 
                                                0.03865,
                                                Elek_price["prijs_excl_belastingen"])


Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20200329) & 
                                                (Elek_price["hour"] == 2), 
                                                0.00885,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20210328) & 
                                                (Elek_price["hour"] == 2), 
                                                0.037,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20220327) & 
                                                (Elek_price["hour"] == 2), 
                                                0.21795,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20230326) & 
                                                (Elek_price["hour"] == 2), 
                                                0.08245,
                                                Elek_price["prijs_excl_belastingen"])

Elek_price["prijs_excl_belastingen"] = np.where((Elek_price["year_mon_day"] == 20240331) & 
                                                (Elek_price["hour"] == 2), 
                                                0.069775,
                                                Elek_price["prijs_excl_belastingen"])

In [249]:
Elek_price["prijs_excl_belastingen"].isnull().sum()

0

# Export to CSV

In [251]:
Elek_price.to_csv("Elek_price.csv")